In [ ]:
import matplotlib.pyplot as plt
# import matplotlib as mpl
import matplotlib.lines as mlines
import networkx as nx
import ndlib.models.ModelConfig as mc

from DiffusionTrend import DiffusionTrend
from SIRVModel import SIRVModel
from sweep import run_sweep
from sweep import run_init_sweep
from sweep import _process_one
from sweep import run_sweep_flex

import numpy as np
import pandas as pd
import seaborn as sns
from time import time
from pathlib import Path

import itertools, os, traceback, sys

from joblib import Parallel, delayed
from multiprocessing import Pool
from random import sample

%matplotlib inline

plt.rcParams['text.usetex'] = True

p = Path('SIRV_Model/data')

### $(C, \eta, \beta, \gamma)$

In [ ]:
# (C, eta, beta, gamma)

# Fix eta and gamma
# param_grid = list(itertools.product(
#     np.arange(0.01, 1.001, 0.01),   # C: 0.01, 0.02, …, 1.0
#     [0.75],                         # eta fixed
#     np.arange(0.5, 1.001, 0.005),   # beta 
#     [0.2]                           # gamma fixed
# ))

# Fix beta and gamma
# param_grid = list(itertools.product(
#     np.arange(0.005, 1.001, 0.005),   # C: 0.1 - 1.0
#     np.arange(0.05, 1.001, 0.1),      # eta
#     [0.8],                            # beta fixed
#     [0.2]                             # gamma fixed
# ))

# th_vals = [0.5, 0.9, 1, 1.1, 2]
th_vals = [0.5]

df_H = run_sweep_flex(
    theta_values=th_vals,
    param_grid=param_grid,
    Dr=0.5, Dg=0.5,
    M=400, N=400,
    epsilon=0.001,
    n_jobs=6
)

In [ ]:
def get_double_plot_fast(data, col='x', game_type='H',
                         max_outer=None,         # 只畫前幾個外層 (gamma, beta)，例如 (3,3)
                         downsample=1,           # 內層 (eta, C) 每隔幾格取一次
                         cmap='jet'):
    data = data[data['Game'] == game_type]
    
    # ---------- 1) 取唯一值 (避免排序開銷過多，必要時才排序) ----------
    outer_x = np.sort(data['beta'].unique())                  # β 升序
    outer_y = np.sort(data['gamma'].unique())[::-1]           # γ 降序
    inner_x = np.sort(data['C'].unique())                     # C
    inner_y = np.sort(data['eta'].unique())                   # η

    B, G, Cn, En = len(outer_x), len(outer_y), len(inner_x), len(inner_y)

    # ---------- 2) 把 (β,γ,η,C) 映射到 4D array 位置，避開 pivot ----------
    # 建類別編碼（O(n)）
    bi = pd.Categorical(data['beta'], categories=outer_x, ordered=True).codes
    gi = pd.Categorical(data['gamma'], categories=outer_y, ordered=True).codes
    ci = pd.Categorical(data['C'],    categories=inner_x, ordered=True).codes
    ei = pd.Categorical(data['eta'],  categories=inner_y, ordered=True).codes

    arr = np.full((G, B, En, Cn), np.nan, dtype=float)
    vals = data[col].to_numpy()
    arr[gi, bi, ei, ci] = vals   # 直接填值

    # ---------- 3) 色階範圍 ----------
    if col == 'payoff_D':
        vmin, vmax = -1.0, 0.0
    elif col == 'payoff_C':
        vmin, vmax = -1.0, 1.0
    else:
        vmin, vmax = 0.0, 1.0

    # ---------- 4) 限縮繪製區域 + 降採樣 ----------
    g_lim = G if not max_outer else min(G, max_outer[0])
    b_lim = B if not max_outer else min(B, max_outer[1])
    # 內層降採樣（為了看 layout/字體很夠用）
    arr_view = arr[:g_lim, :b_lim, ::downsample, ::downsample]
    inner_y_ticks = inner_y[::downsample]
    inner_x_ticks = inner_x[::downsample]

    # ---------- 5) 畫圖：用 imshow（快），關掉多餘 ticks ----------
    fig, axes = plt.subplots(nrows=g_lim, ncols=b_lim,
                             figsize=(20, 20), sharex=False, sharey=False)
    if g_lim == 1 and b_lim == 1:
        axes = np.array([[axes]])
    elif g_lim == 1:
        axes = axes[np.newaxis, :]
    elif b_lim == 1:
        axes = axes[:, np.newaxis]

    fig.subplots_adjust(wspace=0.1, hspace=0.1)

    for i in range(g_lim):
        for j in range(b_lim):
            ax = axes[i, j]
            # 注意：原本你用 [::-1] 讓 η 反序，本質是把 row 上下翻轉。
            # imshow 用 origin='upper' 就能符合「上方是大的 η」的直覺，不需額外複製數據。
            im = ax.imshow(arr_view[i, j], vmin=vmin, vmax=vmax, cmap=cmap,
                           origin='lower', aspect='auto', interpolation='nearest',
                           rasterized=True)

            # 只有左下角保留 tick（且只標 0.1 / 1 的兩端）
            if i != g_lim - 1 and j == 0 and (i % 3 == 0):
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_ylabel('')
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels(["", ""], fontsize=38)
            elif i == g_lim - 1 and j != 0 and (j % 3 == 0):
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_xlabel('')
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels(["", ""], fontsize=38)
            elif i == g_lim - 1 and j == 0:
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_ylabel('')
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_xlabel('')
            else:
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels(["", ""], fontsize=38)
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels(["", ""], fontsize=38)

    # ---------- 6) 外層標籤 + 單一 colorbar ----------
    fig.text(-4.5, -1.5, r'$\beta$', ha='center', va='center', fontsize=96, transform=ax.transAxes)
    fig.text(-11.5, 5.5, r'$\gamma$', ha='center', va='center', rotation=0, fontsize=96, transform=ax.transAxes)
    fig.text(-5, -0.5, r'$C$', ha='center', va='center', fontsize=64, transform=ax.transAxes)
    fig.text(-10.5, 5, r'$\eta$', ha='center', va='center', fontsize=64, transform=ax.transAxes)

    # fig.text(0.16, 0.04, r'$0.1$', ha='center', va='center', fontsize=48)
    # fig.text(0.385, 0.04, r'$0.4$', ha='center', va='center', fontsize=48)
    # fig.text(0.61, 0.04, r'$0.7$', ha='center', va='center', fontsize=48)
    # fig.text(0.83, 0.04, r'$1.0$', ha='center', va='center', fontsize=48)

    # fig.text(0.06, 0.14, r'$0.1$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.375, r'$0.4$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.61, r'$0.7$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.84, r'$1.0$', ha='center', va='center', fontsize=48)

    if col == 'x':
        title_dict = {
            'CH': ('a', 'Chicken'), 
            'PD': ('b', 'Prisoner\'s Dilemma'),
            'H': ('c', 'Harmony'),
            'SH': ('d', 'Stag Hunt'),
            }
    else:
        title_dict = {
            'PD': ('a', 'Prisoner\'s Dilemma'),
            'H': ('b', 'Harmony'),
            }

    title = title_dict[game_type]
    # a, b, c, d
    fig.text(-11.4, 11.4, f"({title[0]})", ha='left', va='bottom', fontsize=64, transform=ax.transAxes)
    # Game type
    fig.text(-4.4, 11.2, fr"\bf {title[1]}", ha='center', va='bottom', fontsize=84, transform=ax.transAxes)


    # 共享 colorbar（用最後一個 im 的 mappable）
    boundaries = np.linspace(0, 1, 21)  # 10個區間
    cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical',
                        boundaries=boundaries,
                        fraction=0.02, aspect=30, pad=0.03)
    tick_positions = np.linspace(0, 1, 6)  # 顯示 5 個主要刻度
    cbar.set_ticks(tick_positions)
    cbar.set_ticklabels([fr'${x:.1f}$' for x in tick_positions])
    cbar.ax.tick_params(labelsize=64, pad=20)

    # ---------- 7) 繪製指定位置的【浮動式】外層刻度線 ---
    outer_tick_len = 0.01  # 刻度線的長度
    outer_tick_gap = 0.02  # 刻度線與圖表邊界的【間距】
    outer_pad = 0.015       # 數字標籤與刻度線的間距

    # --- 外部 X 軸 (beta) 的指定刻度 ---
    target_betas = [0.1, 0.4, 0.7, 1.0]
    for beta_val in target_betas:
        j = np.argmin(np.abs(outer_x - beta_val))
        ax = axes[-1, j]
        pos = ax.get_position()
        x_center = pos.x0 + pos.width / 2
        
        # 線條的起點和終點都向下平移 outer_tick_gap 的距離
        y_start = pos.y0 - outer_tick_gap
        y_end = pos.y0 - outer_tick_gap - outer_tick_len
        line_bottom = mlines.Line2D([x_center, x_center], [y_start, y_end], 
                                    color='black', transform=fig.transFigure)
        fig.add_artist(line_bottom)
        
        # 文字位置也跟著調整
        fig.text(x_center, y_end - outer_pad, fr"${beta_val:.1f}$", 
                 ha='center', va='top', fontsize=64, transform=fig.transFigure)
                 
    # --- 外部 Y 軸 (gamma) 的指定刻度 ---
    target_gammas = [0.1, 0.4, 0.7, 1.0]
    for gamma_val in target_gammas:
        i = np.argmin(np.abs(outer_y - gamma_val))
        ax = axes[i, 0]
        pos = ax.get_position()
        y_center = pos.y0 + pos.height / 2

        # 線條的起點和終點都向左平移 outer_tick_gap 的距離
        x_start = pos.x0 - outer_tick_gap
        x_end = pos.x0 - outer_tick_gap - outer_tick_len
        line_left = mlines.Line2D([x_start, x_end], [y_center, y_center], 
                                  color='black', transform=fig.transFigure)
        fig.add_artist(line_left)

        # 文字位置也跟著調整
        fig.text(x_end - outer_pad, y_center, fr"${gamma_val:.1f}$", 
                 ha='right', va='center', fontsize=64, transform=fig.transFigure)

    plt.show()
    return

# (a) CH (b) PD
# (c) H  (d) SH

game_type = 'PD'

get_double_plot_fast(df, 'x', game_type)

### Cross-section

In [ ]:
def plot_cross_section_with_std(data, dep_var_base, indep_var, fixed_param_sets):
    """
    Plots cross-section data, showing the mean as a line with markers 
    and the standard deviation as a shaded error band.
    """
    fig, ax = plt.subplots(figsize=(25, 20))
    
    mean_col = f"{dep_var_base}_mean"
    std_col = f"{dep_var_base}_std"

    markers = ['o', 'v', 's', 'D', 'h', 'P']
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0, 1, len(fixed_param_sets)))

    for i, fixed_params in enumerate(fixed_param_sets):
        filtered_data = data.copy()
        for param, value in fixed_params.items():
            if param in data.columns:
                filtered_data = filtered_data[filtered_data[param] == value]

        if filtered_data.empty:
            print(f"No data available for {fixed_params}. Skipping.")
            continue

        filtered_data = filtered_data.sort_values(by=indep_var)

        vari = list(fixed_params.items())[-1][0]
        if vari != 'C':
            vari = f"\\{vari}"
        val = list(fixed_params.items())[-1][1]
        label = f"${vari} = {str(val)}$"
        
        # Plot the mean value line
        ax.plot(
            filtered_data[indep_var], 
            filtered_data[mean_col], 
            color=colors[i],
            marker=markers[i % len(markers)], 
            linestyle='-', 
            label=label, 
            lw=4,
            markersize=30,
            mew=1.5,
            mec='black'
        )
        
        # Plot the standard deviation as a shaded area
        ax.fill_between(
            filtered_data[indep_var],
            filtered_data[mean_col] - filtered_data[std_col],
            filtered_data[mean_col] + filtered_data[std_col],
            color=colors[i],
            alpha=0.2
        )
    
    if indep_var == 'C':
        ax.set_xlabel(f'${indep_var}$', fontsize=64, labelpad=30)
    else:
        ax.set_xlabel(f'$\\{indep_var}$', fontsize=64, labelpad=30)
    
    ax.set_ylabel(fr'FOV (${dep_var_base}$)', fontsize=64, labelpad=30)
    ax.legend(fontsize=56, loc='upper left', bbox_to_anchor=(1, 1))
    ax.tick_params(axis='both', which='major', labelsize=56)
    ax.text(-0.05, 1.02, '(d)', fontsize=64, transform=ax.transAxes, ha='left', va='bottom')
    # ax.grid(True, linestyle='--', alpha=0.5)
    
    # fig.tight_layout(rect=[0, 0, 0.85, 1])
    
    plt.show()
    
    return fig, ax

def get_agg_data(data):
    grouped_data = data[cols + ['x']].groupby(cols).agg(['mean', 'std'])
    flat_columns = ['_'.join(col).strip() for col in grouped_data.columns.values]
    grouped_data.columns = flat_columns
    grouped_data = grouped_data.reset_index()
    return grouped_data

df = pd.read_csv(p / 'Final_cross_section' / 'MC_sim.csv')

cols = ['C', 'eta', 'beta', 'gamma']

for col in ['C', 'eta', 'beta', 'gamma', 'theta']:
    df[col] = df[col].round(1)

df_PD = df[df['Game'] == 'PD']
df_H = df[df['Game'] == 'H']

data1 = df_PD[df_PD['run_id'] <= 10]                              # C=0.2, eta=0.8, gamma=0.1,0.3,0.5,0.7,0.9
data2 = df_PD[(df_PD['run_id'] > 10) & (df_PD['run_id'] <= 20)]   # C=0.2, eta=0.8, beta=0.1,0.3,0.5,0.7,0.9
data3 = df_PD[(df_PD['run_id'] > 20) & (df_PD['run_id'] <= 30)]   # beta=0.8, gamma=0.2

fixed_param = 'C'

# plot_cross_section_with_std(
#     data=get_agg_data(data2),
#     dep_var_base='x',
#     indep_var='gamma',
#     fixed_param_sets=[
#         {'C': 0.2, 'eta': 0.8, fixed_param: 0.1},
#         {'C': 0.2, 'eta': 0.8, fixed_param: 0.3},
#         {'C': 0.2, 'eta': 0.8, fixed_param: 0.5},
#         {'C': 0.2, 'eta': 0.8, fixed_param: 0.7},
#         {'C': 0.2, 'eta': 0.8, fixed_param: 0.9}
#     ]
# )

plot_cross_section_with_std(
    data=get_agg_data(data3),
    dep_var_base='x',
    indep_var='eta',
    fixed_param_sets=[
        {'gamma': 0.2, 'beta': 0.8, fixed_param: 0.1},
        {'gamma': 0.2, 'beta': 0.8, fixed_param: 0.3},
        {'gamma': 0.2, 'beta': 0.8, fixed_param: 0.5},
        {'gamma': 0.2, 'beta': 0.8, fixed_param: 0.7},
        {'gamma': 0.2, 'beta': 0.8, fixed_param: 0.9}
    ]
)

### Heat map

In [ ]:
def plot_x0n0_heatmap(
    data: pd.DataFrame,
    *,                    # 之後全都是 keyword‑only
    col: str = 'x',       # 要畫哪一欄
    C: float | None = None,
    eta: float | None = None,
    beta: float | None = None,
    gamma: float | None = None,
    theta: float | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    title: str | None = None,
    figsize=(6, 5),
):
    """
    畫一張 10×10 heat‑map：橫 x0, 縱 n0。
    若指定 C/eta/beta/gamma/theta，則先 filter，再 pivot。
    """
    # -------- 1. 依指定參數篩選資料 ---------------------------
    mask = np.ones(len(data), dtype=bool)
    for name, val in [('C', C), ('eta', eta),
                      ('beta', beta), ('gamma', gamma),
                      ('theta', theta)]:
        if val is not None:
            mask &= np.isclose(data[name], val)
    df = data.loc[mask, ['x0', 'n0', col]]
    # print(df)
    if df.empty:
        raise ValueError("篩選後沒有資料，請確認參數是否正確。")

    # -------- 2. 做 pivot，並補齊 0.1‥1.0 的格點 -------------
    grid = np.round(np.arange(0.1, 1.0 + 1e-8, 0.1), 1)
    pivot = (
        df.pivot(index='n0', columns='x0', values=col)
          .reindex(index=grid, columns=grid)         # 補 NaN
          .sort_index(ascending=False)               # n0 由 1→0.1
    )

    # -------- 3. vmin/vmax 預設 -------------------------------
    if vmin is None or vmax is None:
        if col == 'payoff_D':
            vmin, vmax = -1, 0
        elif col == 'payoff_C':
            vmin, vmax = -1, 1
        else:
            vmin = 0
            vmax = 1

    # -------- 4. 繪圖 ----------------------------------------
    # plt.figure(figsize=figsize)
    fig, ax = plt.subplots(
        figsize=figsize, sharex=False, sharey=False
    )
    sns.heatmap(
        pivot,
        ax=ax,
        cmap='viridis',
        vmin=vmin, vmax=vmax,
        linewidths=0, linecolor='white',  # 讓 NaN 方格邊界明顯
        cbar=False
    )

    sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax),
                               cmap="viridis")
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical',
                        fraction=0.04, pad=0.03)
    cbar.ax.tick_params(labelsize=32)

    # 軸標籤 & tick
    plt.xlabel(r'$x_0$', fontsize=42)
    plt.ylabel(r'$n_0$', fontsize=42, rotation=0, labelpad=15)
    plt.xticks([0.5, 9.5], ['0.1', '1.0'])          # 只留兩端 tick
    plt.yticks([0.5, 9.5], ['1.0', '0.1'])
    ax.set_xticklabels(["0.1", "1"], fontsize=32, rotation=0)
    ax.set_yticklabels(["0.1", "1"], fontsize=32, rotation=0)

    if title:
        plt.title(title, fontsize=42, pad=12)
    plt.tight_layout()
    plt.show()

key_cols = ['x0', 'n0', 'C', 'eta', 'beta', 'gamma']

for col in key_cols:
    df[col] = df[col].round(3)

plot_x0n0_heatmap(df, col='x', C=0.2, eta=0.8, beta=0.8, gamma=0.3, theta=0.5, figsize=(12,10))